In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
from google.colab import files 

uploaded = files.upload()

data = pd.read_csv("classification_data.csv")
x = data[["feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]].values
y = data["label"].values 

df = pd.DataFrame(x, columns = ["feature_1", "feature_2", "feature_3","feature_4", "feature_5"])
df["label"] = y

for i in df.groupby("label"):
    print(i)

asdasdasd
train_list = []
test_list = []

for cls, group in df.groupby("label"):
    group = group.sample(frac = 1, random_state = 42).reset_index(drop = True)
    train_list.append(group.iloc[:350]) # 70% train (350/500)
    test_list.append(group.iloc[350:]) # 30% test (150/500) 

train_df = pd.concat(train_list).reset_index(drop = True)
test_df = pd.concat(test_list).reset_index(drop = True)

feature_pairs = [
    ("feature_1", "feature_2"),
    ("feature_1", "feature_3"),
    ("feature_1", "feature_4"),
    ("feature_1", "feature_5"),
    ("feature_2", "feature_3"),
    ("feature_2", "feature_4"),
    ("feature_2", "feature_5"),
    ("feature_3", "feature_4"),
    ("feature_3", "feature_5"),
    ("feature_4", "feature_5")
]

markers = ["o", "s", "^", "v"]
colors = ["red", "blue", "green", "purple"]

# Vẽ train data
plt.figure(figsize = (20, 16))
for idx, j in enumerate(feature_pairs):
    plt.subplot(5, 2, idx+1)  
    
    for i, cls in enumerate(train_df["label"].unique()):
        subnet = train_df[train_df["label"] == cls]
        plt.scatter(subnet[j[0]], subnet[j[1]], marker=markers[i], color=colors[i], label=f"Train Class {cls}", s=80, edgecolor="black", alpha=0.6)
    
    plt.xlabel(j[0], fontsize=10)
    plt.ylabel(j[1], fontsize=10)
    plt.title(f"Train: {j[0]} vs {j[1]}", fontsize=11)
    plt.legend(fontsize=9)
        
plt.tight_layout()
plt.show()

# Vẽ test data
plt.figure(figsize = (20, 16))
for idx, j in enumerate(feature_pairs):
    plt.subplot(5, 2, idx+1)  
    for i, cls in enumerate(test_df["label"].unique()):
        subnet = test_df[test_df["label"] == cls]
        plt.scatter(subnet[j[0]], subnet[j[1]], marker=markers[i], color=colors[i], label=f"Test Class {cls}", s=100, linewidths=2, alpha=0.6)
    
    plt.xlabel(j[0], fontsize=10)
    plt.ylabel(j[1], fontsize=10)
    plt.title(f"Test: {j[0]} vs {j[1]}", fontsize=11)
    plt.legend(fontsize=9)

plt.tight_layout()
plt.show()


x_train = train_df[["feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]].values
y_train = train_df["label"].values
x_test = test_df[["feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]].values
y_test = test_df["label"].values

x_train_tensor = torch.tensor(x_train, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)
x_test_tensor = torch.tensor(x_test, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

model = nn.Sequential(
    nn.Linear(5, 100),
    nn.Tanh(),
    nn.Linear(100, 4)
)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

epochs = 10000
for epoch in range(epochs):
    outputs = model(x_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch [{epoch}/{epochs}], Loss: {loss.item():.8f}")

with torch.no_grad():
    test_outputs = model(x_test_tensor)
    predicted = torch.argmax(test_outputs, dim=1)
    accuracy = (predicted == y_test_tensor).sum().item() / len(y_test)
    print(f"Test outputs: {test_outputs}")
    print(f"Predicted classes: {predicted}")
    print("True classes:   ", y_test_tensor)
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

# 5. Vẽ Confusion Matrix
y_pred_np = predicted.cpu().numpy()
y_test_np = y_test_tensor.cpu().numpy()

cm = confusion_matrix(y_test_np, y_pred_np)
print("\nConfusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                            display_labels=['Class 0', 'Class 1', 'Class 2', 'Class 3'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - Best Model')
plt.show()

# Nhận xét
print("\nNhận xét Confusion Matrix:")
print(f"- Tổng số mẫu test: {np.sum(cm)}")
print(f"- Số mẫu dự đoán đúng: {np.trace(cm)} (trên đường chéo chính)")
print(f"- Số mẫu dự đoán sai: {np.sum(cm) - np.trace(cm)}")
for i in range(4):
    print(f"- Class {i}: {cm[i,i]}/{np.sum(cm[i,:])} dự đoán đúng ({cm[i,i]/np.sum(cm[i,:])*100:.2f}%)")